# vocab.registry
> Typed, query-able vocabulary registry for CogitareLink.
---
Loads `data/registry_data.json` at import-time, validates each
entry with `pydantic`, and exposes convenient search helpers.

In [ ]:
#| default_exp vocab.registry

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import json, importlib.resources as pkg
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, AnyUrl, Field, ConfigDict, ValidationError

from cogitarelink.core.debug import get_logger
from urllib.parse import urlparse, urlunparse




In [ ]:
#| export
log = get_logger("registry")

# Load bundled JSON once
_RAW_JSON: Dict[str, Any] = json.loads(
    pkg.files("cogitarelink").joinpath("data/registry_data.json").read_text()
)

## VocabEntry

In [ ]:
#| export
class VocabEntry(BaseModel):
    uri: AnyUrl
    alternative_uris: List[AnyUrl] = []
    prefix: str
    title: str
    description: str
    version: str
    publisher: str
    support_level: str            # "direct" | "cache" | "github_raw" etc.
    resources: Dict[str, Any]
    access_patterns: Dict[str, Any]
    url_transformations: List[Dict[str, str]] = []
    features: Dict[str, bool]
    common_terms: List[str] = Field(default_factory=list)
    common_types: List[str] = Field(default_factory=list)
    related_vocabs: List[str] = Field(default_factory=list)

In [ ]:
#| export
class ResourceURLs(BaseModel):
    """
    Canonical set of resource links that *may* exist for a vocabulary.
    Only `context` **or** `ttl` is required; others are optional.
    """
    ttl:     Optional[AnyUrl] = None   # Turtle serialisation
    context: Optional[AnyUrl] = None   # JSON-LD context (if published)
    backup:  Optional[AnyUrl] = None   # Alternate mirror / raw Git link
    homepage:Optional[AnyUrl] = None


In [ ]:
#| export
class VocabEntry(BaseModel):
    """
    Single vocabulary description used by CogitareLink.

    • All URLs are validated/well-formed (`AnyUrl`)  
    • Optional helper properties (`has_context`, `all_uris`) make
      client code cleaner.
    """
    # ---- identity ---------------------------------------------------
    uri:              AnyUrl                 # canonical root IRI
    alternative_uris: List[AnyUrl] = Field(default_factory=list)
    prefix:           str

    # ---- human info -------------------------------------------------
    title:       str
    description: str
    version:     str
    publisher:   str

    # ---- runtime hints ----------------------------------------------
    support_level: str                       # "direct" | "cache" | "github_raw" | ...
    derives_context: bool = False            # we must synthesize context from TTL?

    # ---- resource & access metadata ---------------------------------
    resources:         ResourceURLs
    access_patterns:   Dict[str, List[str]]  # primary/fallback strategy keywords
    url_transformations: List[Dict[str, str]] = Field(default_factory=list)

    # ---- JSON-LD feature flags --------------------------------------
    features: Dict[str, bool]                # inline_context, uses_protection, ...

    # ---- common terms for quick-autocomplete ------------------------
    common_terms:  List[str] = Field(default_factory=list)
    common_types:  List[str] = Field(default_factory=list)
    related_vocabs:List[str] = Field(default_factory=list)

    # ---- Pydantic config --------------------------------------------
    model_config = ConfigDict(
        extra="forbid",
        validate_default=True,
        frozen=True              # make instances hashable / cache-safe
    )

    # ---- helper properties ------------------------------------------
    def has_context(self) -> bool:
        "True if the registry entry supplies a JSON-LD context URL."
        return bool(self.resources.context)

    @property
    def all_uris(self) -> List[str]:
        "Canonical + alternative URIs as plain strings."
        return [str(self.uri), *[str(u) for u in self.alternative_uris]]

In [ ]:
#| export
def _norm(u:str) -> str:
    """Lower-case scheme+host and drop a single trailing '/'."""
    p = urlparse(u)
    scheme, netloc, path = p.scheme.lower(), p.netloc.lower(), p.path
    if path.endswith("/") and path != "/":
        path = path[:-1]
    return urlunparse((scheme, netloc, path, "", "", ""))


In [ ]:
#| export
class VRegistry:
    "Immutable mapping of `prefix -> VocabEntry`."
    def __init__(self, raw: Dict[str, Any] | None = None):
        raw = raw or _RAW_JSON
        self._data: Dict[str, VocabEntry] = {}
        invalid: Dict[str, str] = {}
        for k, v in raw.items():
            try:
                self._data[k] = VocabEntry(**v)
            except ValidationError as e:
                invalid[k] = e.json()
        if invalid:
            log.warning("Skipped %d invalid vocab entries", len(invalid))
        # build once after all valid entries were loaded
        def _to_str_list(entry):
            "Return main + alt URIs as plain strings"
            return [str(entry.uri), *[str(u) for u in entry.alternative_uris]]

        self._alt_uri_map = {
            _norm(url): entry
            for entry in self._data.values()
            for url   in _to_str_list(entry)
        }
        
    # ------------- basic look-ups ----------------------------------
    def by_prefix(self, p: str) -> VocabEntry:
        """Get vocabulary entry by prefix.
        
        Args:
            p: The prefix to look up
            
        Returns:
            The vocabulary entry
            
        Raises:
            KeyError: If the prefix is not found in the registry
        """
        try:
            return self._data[p]
        except KeyError:
            raise KeyError(f"Vocabulary prefix '{p}' not found in registry.")
    
    def by_uri(self, uri: str) -> VocabEntry:
        """Get vocabulary entry by URI.
        
        Args:
            uri: The URI to look up
            
        Returns:
            The vocabulary entry
            
        Raises:
            KeyError: If the URI is not found in the registry
        """
        normalized = _norm(uri)
        try:
            return self._alt_uri_map[normalized]
        except KeyError:
            raise KeyError(f"URI '{uri}' not found in registry.")
    
    def get_by_prefix(self, p: str, default=None) -> VocabEntry | None:
        """Get vocabulary entry by prefix, returning default if not found.
        
        Args:
            p: The prefix to look up
            default: Value to return if prefix is not found
            
        Returns:
            The vocabulary entry or default
        """
        return self._data.get(p, default)
    
    def get_by_uri(self, uri: str, default=None) -> VocabEntry | None:
        """Get vocabulary entry by URI, returning default if not found.
        
        Args:
            uri: The URI to look up
            default: Value to return if URI is not found
            
        Returns:
            The vocabulary entry or default
        """
        normalized = _norm(uri)
        return self._alt_uri_map.get(normalized, default)
    
    def search(self, kw: str) -> List[VocabEntry]:
        """Search for vocabularies containing keyword in title or description.
        
        Args:
            kw: Keyword to search for
            
        Returns:
            List of matching vocabulary entries
        """
        kw = kw.lower()
        return [v for v in self._data.values()
                if kw in v.description.lower() or kw in v.title.lower()]

    # ------------- dunder helpers ----------------------------------
    def __iter__(self): return iter(self._data)
    def __getitem__(self, k): return self.by_prefix(k)  # Use improved by_prefix
    def __len__(self): return len(self._data)
    def __contains__(self, k): return k in self._data

# global singleton
registry = VRegistry()


W|cogitarelink.registry|W|cogitarelink.registry|Skipped 8 invalid vocab entries


In [ ]:
#| hide
from fastcore.test import test, test_eq, test_ne, test_is, test_fail
from operator import eq
#from cogitarelink.vocab.registry import registry, VocabEntry

# Basic functionality tests
test_is(isinstance(registry["schema"], VocabEntry), True)
test(len(registry) > 5, True, eq, "registry should have >5 vocabs")

# Get entry by prefix
schema = registry.by_prefix("schema")
test_eq(schema.prefix, "schema")
test_eq(schema.resources["context"].split("/")[2], "schema.org")

# Get entry by URI
alt = registry.by_uri("http://schema.org/")
test_is(alt, schema)

# Search functionality
hits = registry.search("supply chain")
test_ne(len(hits), 0)
test_eq(any(v.prefix == "epcis" for v in hits), True)

# Test error handling for non-existent prefix
try:
    registry.by_prefix("no-such-vocab")
    assert False, "Should have raised KeyError"
except KeyError as e:
    assert "not found in registry" in str(e), "Error message should be informative"

# Test error handling for non-existent URI
try:
    registry.by_uri("http://example.org/nonexistent")
    assert False, "Should have raised KeyError"
except KeyError as e:
    assert "not found in registry" in str(e), "Error message should be informative"

# Test get_by_prefix with default
test_is(registry.get_by_prefix("no-such-vocab"), None)
test_eq(registry.get_by_prefix("no-such-vocab", "default"), "default")

# Test get_by_uri with default
test_is(registry.get_by_uri("http://example.org/nonexistent"), None)
test_eq(registry.get_by_uri("http://example.org/nonexistent", "default"), "default")

# Test contains
test_eq("schema" in registry, True)
test_eq("no-such-vocab" in registry, False)

print("All tests passed!")


KeyError: "Vocabulary prefix 'schema' not found in registry."

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

/Users/cvardema/dev/git/LA3D/cogitarelink/.venv/lib/python3.11/site-packages/nbdev/export.py:88: UserWarning: Notebook '/Users/cvardema/dev/git/LA3D/cogitarelink/scratch.ipynb' uses `#|export` without `#|default_exp` cell.
Note nbdev2 no longer supports nbdev1 syntax. Run `nbdev_migrate` to upgrade.
See https://nbdev.fast.ai/getting_started.html for more information.
  warn(f"Notebook '{nbname}' uses `#|export` without `#|default_exp` cell.\n"
